In [20]:
from gs_lib.gs_tools import *

def make_participants():
    m1, m2, m3, m4 = Man("m1"), Man("m2"), Man("m3"), Man("m4")
    w1, w2, w3, w4 = Woman("w1"), Woman("w2"), Woman("w3"), Woman("w4")
    return (m1, m2, m3, m4), (w1, w2, w3, w4)

def make_preferences(men, women):
    m1, m2, m3, m4 = men
    w1, w2, w3, w4 = women
    return PreferenceList({
        m1: [w1, w2, w3, w4],
        m2: [w2, w1, w4, w3],
        m3: [w3, w4, w1, w2],
        m4: [w4, w3, w2, w1],
        w1: [m4, m3, m2, m1],
        w2: [m3, m4, m1, m2],
        w3: [m2, m1, m4, m3],
        w4: [m1, m2, m3, m4],
    })


men, women = make_participants()
prefs = make_preferences(men, women)

In [21]:
print("=" * 70)
print("DEMO: BUILDING SIDES AND EXPRESSING MATCHINGS")
print("=" * 70)

# ── 1. CREATE THE TWO SIDES ──────────────────────────
print("\n1. CREATING MEN'S SIDE AND WOMEN'S SIDE")
print("-" * 40)

men = [Man("Adam"), Man("Bob"), Man("Charlie"), Man("David")]
women = [Woman("Eve"), Woman("Fiona"), Woman("Grace"), Woman("Helen")]

print(f"   Men:   {', '.join(m.id for m in men)}")
print(f"   Women: {', '.join(w.id for w in women)}")

# ── 2. EXPRESS PREFERENCES ───────────────────────────
print("\n2. EXPRESSING PREFERENCES")
print("-" * 40)

prefs = PreferenceList({
    # Men's preferences (1st > 2nd > 3rd > 4th)
    men[0]: [women[0], women[1], women[2], women[3]],  # Adam
    men[1]: [women[1], women[0], women[3], women[2]],  # Bob
    men[2]: [women[2], women[3], women[0], women[1]],  # Charlie
    men[3]: [women[3], women[2], women[1], women[0]],  # David
    
    # Women's preferences (1st > 2nd > 3rd > 4th)
    women[0]: [men[3], men[2], men[1], men[0]],        # Eve
    women[1]: [men[2], men[3], men[0], men[1]],        # Fiona
    women[2]: [men[1], men[0], men[3], men[2]],        # Grace
    women[3]: [men[0], men[1], men[2], men[3]],        # Helen
})

print("\n   Men's preferences:")
for m in men:
    ranked = " > ".join(w.id for w in prefs.get_preference(m))
    print(f"     {m.id}: {ranked}")

print("\n   Women's preferences:")
for w in women:
    ranked = " > ".join(m.id for m in prefs.get_preference(w))
    print(f"     {w.id}: {ranked}")

DEMO: BUILDING SIDES AND EXPRESSING MATCHINGS

1. CREATING MEN'S SIDE AND WOMEN'S SIDE
----------------------------------------
   Men:   Adam, Bob, Charlie, David
   Women: Eve, Fiona, Grace, Helen

2. EXPRESSING PREFERENCES
----------------------------------------

   Men's preferences:
     Adam: Eve > Fiona > Grace > Helen
     Bob: Fiona > Eve > Helen > Grace
     Charlie: Grace > Helen > Eve > Fiona
     David: Helen > Grace > Fiona > Eve

   Women's preferences:
     Eve: David > Charlie > Bob > Adam
     Fiona: Charlie > David > Adam > Bob
     Grace: Bob > Adam > David > Charlie
     Helen: Adam > Bob > Charlie > David


In [22]:
# ── 3. RUN GALE-SHAPLEY ──────────────────────────────
print("\n3. RUNNING GALE-SHAPLEY ALGORITHM")
print("-" * 40)

gs = GaleShapley(prefs)
man_opt, woman_opt = gs.find_both_optimal()

print(f"\n   Man-optimal matching:")
print(f"     {man_opt}")
print("     Satisfaction:")
for m in men:
    w = man_opt.get_partner(m)
    rank = prefs.get_rank(m, w) + 1
    print(f"       {m.id} → {w.id} (his #{rank} choice)")

print(f"\n   Woman-optimal matching:")
print(f"     {woman_opt}")
print("     Satisfaction:")
for w in women:
    m = woman_opt.get_partner(w)
    rank = prefs.get_rank(w, m) + 1
    print(f"       {w.id} → {m.id} (her #{rank} choice)")


3. RUNNING GALE-SHAPLEY ALGORITHM
----------------------------------------

   Man-optimal matching:
     Matching(pairs=[(Adam-Eve), (Bob-Fiona), (Charlie-Grace), (David-Helen)], unmatched_men=[], unmatched_women=[])
     Satisfaction:
       Adam → Eve (his #1 choice)
       Bob → Fiona (his #1 choice)
       Charlie → Grace (his #1 choice)
       David → Helen (his #1 choice)

   Woman-optimal matching:
     Matching(pairs=[(Adam-Helen), (Bob-Grace), (Charlie-Fiona), (David-Eve)], unmatched_men=[], unmatched_women=[])
     Satisfaction:
       Eve → David (her #1 choice)
       Fiona → Charlie (her #1 choice)
       Grace → Bob (her #1 choice)
       Helen → Adam (her #1 choice)


In [23]:
man_optimal = gs.find_stable_matching("men")
print(man_optimal)

Matching(pairs=[(Adam-Eve), (Bob-Fiona), (Charlie-Grace), (David-Helen)], unmatched_men=[], unmatched_women=[])


In [24]:
man_optimal = gs.find_stable_matching("women")
print(man_optimal)

Matching(pairs=[(Adam-Helen), (Bob-Grace), (Charlie-Fiona), (David-Eve)], unmatched_men=[], unmatched_women=[])


In [25]:
# Unequal two-sides:

m1u, m2u, m3u = Man("m1"), Man("m2"), Man("m3")
w1u, w2u = Woman("w1"), Woman("w2")

unequal = PreferenceList({
    m1u: [w1u, w2u],
    m2u: [w2u, w1u],
    m3u: [w1u],
    w1u: [m2u, m1u, m3u],
    w2u: [m1u, m2u],
})

gs_u = GaleShapley(unequal)
result = gs_u.find_stable_matching("men")
print(result)
# One man will end up in unmatched_men
print("Unmatched men:", result.unmatched_men)

Matching(pairs=[(m1-w1), (m2-w2)], unmatched_men=[m3], unmatched_women=[])
Unmatched men: frozenset({Man(m3)})


In [26]:
verifier = StabilityVerifier(prefs)
ok, reason, blocking = verifier.is_stable(man_optimal)
print(ok)          # True
print(reason)      # None
print(blocking)    # None

True
None
None


In [27]:
m1, m2, m3, m4 = Man("m1"), Man("m2"), Man("m3"), Man("m4")
w1, w2, w3, w4 = Woman("w1"), Woman("w2"), Woman("w3"), Woman("w4")

prefs = PreferenceList({
    m1: [w1, w2, w3, w4],
    m2: [w2, w1, w4, w3],
    m3: [w3, w4, w1, w2],
    m4: [w4, w3, w2, w1],
    w1: [m4, m3, m2, m1],
    w2: [m3, m4, m1, m2],
    w3: [m2, m1, m4, m3],
    w4: [m1, m2, m3, m4],
})

all_men   = {m1, m2, m3, m4}
all_women = {w1, w2, w3, w4}

verifier = StabilityVerifier(prefs)

In [28]:
reverse = Matching.from_dict(
    {m1: w4, m2: w3, m3: w2, m4: w1},
    all_men, all_women,
)
verifier.is_stable(reverse)

(True, None, None)

In [29]:


# --- Setup -------------------------------------------------------------------
m1, m2, m3, m4 = Man("m1"), Man("m2"), Man("m3"), Man("m4")
w1, w2, w3, w4 = Woman("w1"), Woman("w2"), Woman("w3"), Woman("w4")

prefs = PreferenceList({
    m1: [w1, w2, w3, w4],
    m2: [w2, w1, w4, w3],
    m3: [w3, w4, w1, w2],
    m4: [w4, w3, w2, w1],
    w1: [m4, m3, m2, m1],
    w2: [m3, m4, m1, m2],
    w3: [m2, m1, m4, m3],
    w4: [m1, m2, m3, m4],
})

all_men   = {m1, m2, m3, m4}
all_women = {w1, w2, w3, w4}

verifier = StabilityVerifier(prefs)


def check(label, matching):
    print("=" * 70)
    print(label)
    print("=" * 70)
    print(f"Matching: {matching}")
    ok, reason, blocking = verifier.is_stable(matching)
    if ok:
        print("Result:   ✓ STABLE")
    else:
        print("Result:   ✗ UNSTABLE")
        print(f"Reason:   {reason}")
        if blocking:
            man, woman = blocking
            print(f"Blocking: ({man.id}, {woman.id})")
    print()
    return ok


# --- CASE 1: identity matching ----------------------------------------------
identity = Matching.from_dict(
    {m1: w1, m2: w2, m3: w3, m4: w4},
    all_men, all_women,
)
check("CASE 1: Identity matching", identity)


# --- CASE 2: reverse matching -----------------------------------------------
reverse = Matching.from_dict(
    {m1: w4, m2: w3, m3: w2, m4: w1},
    all_men, all_women,
)
check("CASE 2: Reverse matching (m1-w4, m2-w3, m3-w2, m4-w1)", reverse)


# --- CASE 3: hand-picked, likely unstable -----------------------------------
handpicked = Matching.from_dict(
    {m1: w3, m2: w1, m3: w4, m4: w2},
    all_men, all_women,
)
check("CASE 3: Hand-picked (m1-w3, m2-w1, m3-w4, m4-w2)", handpicked)


# --- CASE 4: leave a man unmatched (valid when unequal, but here it's odd) --
partial = Matching.from_dict(
    {m1: w1, m2: w2, m3: w3, m4: None},   # m4 unmatched, w4 unmatched
    all_men, all_women,
)
check("CASE 4: m4 left unmatched", partial)


# --- CASE 5: individual-rationality violation -------------------------------
# m1's list has NO w4. Matching him to w4 should fail immediately.
irrational = Matching.from_dict(
    {m1: w4, m2: w1, m3: w2, m4: w3},
    all_men, all_women,
)
check("CASE 5: m1 matched to unacceptable w4", irrational)


# --- CASE 6: the actual man-optimal (should verify stable) ------------------
from stable_matching import GaleShapley
man_optimal = GaleShapley(prefs).find_stable_matching("men")
check("CASE 6: Man-optimal from Gale-Shapley", man_optimal)


# --- CASE 7: a matching that is stable but NOT produced by GS ---------------
# For this instance, we can search the lattice for a non-extreme node.
from stable_matching import StableMatchingLattice
lattice = StableMatchingLattice(prefs)
lattice.build_lattice()
all_stable = lattice.get_all_matchings()
non_extreme = [m for m in all_stable
               if m != man_optimal
               and m != GaleShapley(prefs).find_stable_matching("women")]
if non_extreme:
    check("CASE 7: A middle stable matching from the lattice", non_extreme[0])
else:
    print("CASE 7: no middle matching in this instance\n")

CASE 1: Identity matching
Matching: Matching(pairs=[(m1-w1), (m2-w2), (m3-w3), (m4-w4)], unmatched_men=[], unmatched_women=[])
Result:   ✓ STABLE

CASE 2: Reverse matching (m1-w4, m2-w3, m3-w2, m4-w1)
Matching: Matching(pairs=[(m1-w4), (m2-w3), (m3-w2), (m4-w1)], unmatched_men=[], unmatched_women=[])
Result:   ✓ STABLE

CASE 3: Hand-picked (m1-w3, m2-w1, m3-w4, m4-w2)
Matching: Matching(pairs=[(m1-w3), (m2-w1), (m3-w4), (m4-w2)], unmatched_men=[], unmatched_women=[])
Result:   ✓ STABLE

CASE 4: m4 left unmatched
Matching: Matching(pairs=[(m1-w1), (m2-w2), (m3-w3)], unmatched_men=[m4], unmatched_women=[w4])
Result:   ✗ UNSTABLE
Reason:   Blocking pair: (m4, w4)
Blocking: (m4, w4)

CASE 5: m1 matched to unacceptable w4
Matching: Matching(pairs=[(m1-w4), (m2-w1), (m3-w2), (m4-w3)], unmatched_men=[], unmatched_women=[])
Result:   ✗ UNSTABLE
Reason:   Blocking pair: (m3, w1)
Blocking: (m3, w1)



ModuleNotFoundError: No module named 'stable_matching'

In [30]:
partial = Matching.from_dict(
    {m1: w1, m2: w2, m3: w3, m4: None},   # m4 unmatched, w4 unmatched
    all_men, all_women,
)
verifier.is_stable(partial)

(False, 'Blocking pair: (m4, w4)', (Man(m4), Woman(w4)))

In [31]:
# diag.py
import random
import numpy as np
from gs_lib.gs_tools import (
    Man, Woman, PreferenceList, GaleShapley, StabilityVerifier,
)
from p2etg import P2ETG


def random_theta(participants, partners, rng, alpha=0.5):
    K = len(partners)
    return {
        p: {q: float(w) for q, w in zip(partners, rng.dirichlet([alpha] * K))}
        for p in participants
    }


def gs_on_true_preferences(men, women, ttm, ttw):
    prefs = {
        **{m: sorted(women, key=lambda w: -ttm[m][w]) for m in men},
        **{w: sorted(men, key=lambda m: -ttw[w][m]) for w in women},
    }
    return GaleShapley(PreferenceList(prefs)).find_stable_matching("men")


def main():
    N, K, seed, alpha = 3, 3, 0, 0.5
    max_epochs = 40

    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    men = [Man(f"m{i}") for i in range(1, N + 1)]
    women = [Woman(f"w{j}") for j in range(1, K + 1)]

    true_theta_men   = random_theta(men, women, np_rng, alpha)
    true_theta_women = random_theta(women, men, np_rng, alpha)

    h_star = gs_on_true_preferences(men, women, true_theta_men, true_theta_women)
    print("H_* pairs:", [(p.man.id, p.woman.id) for p in h_star.pairs])

    learner = P2ETG(
        men, women,
        true_theta_men=true_theta_men,
        true_theta_women=true_theta_women,
        rng=rng,
    )
    result = learner.run_until_stop(
        max_epochs=max_epochs,
        adaptive=True,
        check_every=500,
        max_samples=200_000,
    )

    committed = result["matching"]
    print("committed pairs:", [(p.man.id, p.woman.id) for p in committed.pairs])

    # ---- Diagnostics ----
    prefs_true = PreferenceList({
        **{m: sorted(women, key=lambda w: -true_theta_men[m][w]) for m in men},
        **{w: sorted(men, key=lambda m: -true_theta_women[w][m]) for w in women},
    })
    prefs_hat = learner._build_preference_lists()

    print("\nH_* stable under truth:",
          StabilityVerifier(prefs_true).is_stable(h_star)[0])
    print("H_t stable under truth:",
          StabilityVerifier(prefs_true).is_stable(committed)[0])
    print("H_t stable under ≻̂:",
          StabilityVerifier(prefs_hat).is_stable(committed)[0])

    print("\nPreference comparison (≻̂ vs ≻):")
    for m in men:
        hat  = prefs_hat.get_preference(m)
        true = prefs_true.get_preference(m)
        print(f"  {m}: ≻̂={[str(x) for x in hat]}  "
              f"≻={[str(x) for x in true]}  equal={hat == true}")
    for w in women:
        hat  = prefs_hat.get_preference(w)
        true = prefs_true.get_preference(w)
        print(f"  {w}: ≻̂={[str(x) for x in hat]}  "
              f"≻={[str(x) for x in true]}  equal={hat == true}")

    print("\nθ̂ and counts (first man):")
    state = learner.agent_states[men[0]]
    print(f"  θ̂ = {state.theta}")
    print(f"  counts = {dict(state.counts.total)}")
    print(f"  ci = {state.ci}")


if __name__ == "__main__":
    main()

H_* pairs: [('m2', 'w3'), ('m1', 'w2'), ('m3', 'w1')]
  ep        R      new          t   disj   maxWid   minGap   worstOv  matching
------------------------------------------------------------------------------
   0      500      500        500  False   0.9354   0.0323   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   1      500      500       1000  False   0.7024   0.0179   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   2      500      500       1500  False   0.5937   0.0060   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   3      500      500       2000  False   0.5234   0.0268   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   4      500      500       2500  False   0.4762   0.0036   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   5      500      500      

In [34]:
# diag.py
import random
import numpy as np
from gs_lib.gs_tools import (
    Man, Woman, PreferenceList, GaleShapley, StabilityVerifier,
)
from p2etg import P2ETG


def random_theta(participants, partners, rng, alpha=0.5):
    K = len(partners)
    return {
        p: {q: float(w) for q, w in zip(partners, rng.dirichlet([alpha] * K))}
        for p in participants
    }


def gs_on_true_preferences(men, women, ttm, ttw):
    prefs = {
        **{m: sorted(women, key=lambda w: -ttm[m][w]) for m in men},
        **{w: sorted(men,   key=lambda m: -ttw[w][m]) for w in women},
    }
    return GaleShapley(PreferenceList(prefs)).find_stable_matching("men")


def main():
    N, K, seed, alpha = 3, 3, 0, 0.5

    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    men   = [Man(f"m{i}")   for i in range(1, N + 1)]
    women = [Woman(f"w{j}") for j in range(1, K + 1)]

    true_theta_men   = random_theta(men,   women, np_rng, alpha)
    true_theta_women = random_theta(women, men,   np_rng, alpha)

    h_star = gs_on_true_preferences(men, women, true_theta_men, true_theta_women)

    learner = P2ETG(
        men, women,
        true_theta_men=true_theta_men,
        true_theta_women=true_theta_women,
        rng=rng,
    )
    result = learner.run_until_stop(
        max_epochs=40,
        adaptive=True,
        check_every=500,
        max_samples=200_000,
    )

    # ---- now `learner`, `result`, `h_star`, `men`, `women`,
    #      `true_theta_men`, `true_theta_women` are all in scope ----

    # 1. Sampler orientation check
    flipped = [(agent, key, b1, b2)
               for (agent, key, b1, b2) in learner._arms
               if key[0] != b1]
    print(f"Arms with flipped orientation: {len(flipped)}/{len(learner._arms)}")
    for entry in flipped[:5]:
        print("  ", entry)

    # 2. Is H_t the same as H_*?
    print(f"H_t == H_*: {result['matching'] == h_star}")
    print(f"H_t pairs: {[(p.man.id, p.woman.id) for p in result['matching'].pairs]}")
    print(f"H_* pairs: {[(p.man.id, p.woman.id) for p in h_star.pairs]}")

    # 3. Stability under truth and estimate
    prefs_true = PreferenceList({
        **{m: sorted(women, key=lambda w: -true_theta_men[m][w]) for m in men},
        **{w: sorted(men,   key=lambda m: -true_theta_women[w][m]) for w in women},
    })
    prefs_hat = learner._build_preference_lists()

    print("H_t stable under truth:",
          StabilityVerifier(prefs_true).is_stable(result["matching"])[0])
    print("H_t stable under ≻̂:",
          StabilityVerifier(prefs_hat).is_stable(result["matching"])[0])

    # 4. Preference comparison
    print("\nPreference comparison (≻̂ vs ≻):")
    for p in men + women:
        hat  = prefs_hat.get_preference(p)
        true = prefs_true.get_preference(p)
        print(f"  {p}: equal={hat == true}")
        if hat != true:
            print(f"    ≻̂: {[str(x) for x in hat]}")
            print(f"    ≻:  {[str(x) for x in true]}")


if __name__ == "__main__":
    main()

  ep        R      new          t   disj   maxWid   minGap   worstOv  matching
------------------------------------------------------------------------------
   0      500      500        500  False   0.9354   0.0323   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   1      500      500       1000  False   0.7024   0.0179   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   2      500      500       1500  False   0.5937   0.0060   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   3      500      500       2000  False   0.5234   0.0268   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   4      500      500       2500  False   0.4762   0.0036   0.50000  Matching(pairs=[(m1-w2), (m2-w3), (m3-w1)], unmatched_men=[], unmatched_women=[])
   5      500      500       3000  False   0.4392   0.0269   0.50000  Matching(pai

In [36]:
"""
diag_mle.py
Deep diagnostic for degenerate Bradley-Terry MLE.

Checks:
    1. Whether counts.total and counts.wins have matching keys.
    2. Whether w_i and n_i computed inside bt_mle_mm have signal.
    3. Whether theta_hat is uniform, stuck, or diverging.
    4. Whether the empirical ranking matches the truth.
    5. Whether the arms in P2ETG have consistent orientation.

Run:
    python diag_mle.py
"""

from __future__ import annotations

import random
import numpy as np
import pandas as pd

from gs_lib.gs_tools import (
    Man, Woman, PreferenceList, GaleShapley, StabilityVerifier,
)
from gs_lib.bt import (
    _canonical, PairCounts, bt_mle_mm, bt_ranking
)
from p2etg import P2ETG


# ============================================================================
# Helpers
# ============================================================================

def random_theta(participants, partners, rng, alpha=0.5):
    K = len(partners)
    return {
        p: {q: float(w) for q, w in zip(partners, rng.dirichlet([alpha] * K))}
        for p in participants
    }


def gs_on_true_preferences(men, women, ttm, ttw):
    prefs = {
        **{m: sorted(women, key=lambda w: -ttm[m][w]) for m in men},
        **{w: sorted(men,   key=lambda m: -ttw[w][m]) for w in women},
    }
    return GaleShapley(PreferenceList(prefs)).find_stable_matching("men")


def truth_ranking(state, true_theta_men, true_theta_women):
    """Return the true preference order for the agent of this state."""
    if isinstance(state.agent, Man):
        tt = true_theta_men[state.agent]
    else:
        tt = true_theta_women[state.agent]
    return sorted(state.partners, key=lambda p: -tt[p])


# ============================================================================
# Section 1 — inspect raw counts
# ============================================================================

def inspect_counts(learner):
    print("\n" + "=" * 78)
    print("SECTION 1 — Raw counts and win keys per agent")
    print("=" * 78)

    for state in learner.agent_states.values():
        totals = state.counts.total
        wins = state.counts.wins

        totals_keys = set(totals.keys())
        wins_keys = set(wins.keys())

        print(f"\n[{state.agent}]")
        print(f"  totals: {dict(totals)}")
        print(f"  wins:   {dict(wins)}")
        print(f"  totals - wins: {totals_keys - wins_keys}")
        print(f"  wins - totals: {wins_keys - totals_keys}")

        # Sanity: wins[key] <= totals[key] must hold
        bad = [(k, wins.get(k, 0), totals[k])
               for k in totals_keys
               if wins.get(k, 0) > totals[k]]
        if bad:
            print(f"  !! wins > totals on pairs: {bad}")


# ============================================================================
# Section 2 — recompute w_i, n_i the way bt_mle_mm does
# ============================================================================

def inspect_mle_inputs(learner):
    print("\n" + "=" * 78)
    print("SECTION 2 — Recompute w_i, n_i as bt_mle_mm would")
    print("=" * 78)

    for state in learner.agent_states.values():
        items = state.partners
        totals = state.counts.total
        wins = state.counts.wins

        n_i = {i: 0 for i in items}
        w_i = {i: 0 for i in items}

        for (b1, b2), t in totals.items():
            w12 = wins.get((b1, b2), 0)
            w21 = t - w12
            n_i[b1] += t
            n_i[b2] += t
            w_i[b1] += w12
            w_i[b2] += w21

        print(f"\n[{state.agent}]")
        print(f"  n_i = {n_i}")
        print(f"  w_i = {w_i}")

        # Signal check
        if len(set(n_i.values())) == 1 and len(set(w_i.values())) == 1:
            print("  !! WARNING: n_i and w_i are constant across items — no signal")
        elif len(set(w_i.values())) == 1:
            print("  !! WARNING: w_i is constant across items — MLE has no ordering info")
        else:
            print(f"  w_i range = {max(w_i.values()) - min(w_i.values())}")
            print(f"  w_i ratio (max/min) = "
                  f"{max(w_i.values()) / max(min(w_i.values()), 1)}")


# ============================================================================
# Section 3 — trace bt_mle_mm iterations
# ============================================================================

def trace_mle(learner, max_iter=5):
    print("\n" + "=" * 78)
    print("SECTION 3 — Trace MM iterations for each agent")
    print("=" * 78)

    for state in learner.agent_states.values():
        items = state.partners
        totals = state.counts.total
        wins = state.counts.wins

        print(f"\n[{state.agent}]")
        theta = {i: 1.0 / len(items) for i in items}
        print(f"  init:  {[f'{theta[i]:.6f}' for i in items]}")

        for it in range(max_iter):
            n_i = {i: 0 for i in items}
            w_i = {i: 0 for i in items}
            for (b1, b2), t in totals.items():
                w12 = wins.get((b1, b2), 0)
                n_i[b1] += t
                n_i[b2] += t
                w_i[b1] += w12
                w_i[b2] += t - w12

            denom = {i: 0.0 for i in items}
            for (b1, b2), t in totals.items():
                s = theta[b1] + theta[b2]
                if s <= 0:
                    s = 1e-12
                denom[b1] += t / s
                denom[b2] += t / s

            new_theta = {}
            for i in items:
                if denom[i] <= 0 or w_i[i] <= 0:
                    new_theta[i] = max(theta[i], 1e-12)
                else:
                    new_theta[i] = w_i[i] / denom[i]
            Z = sum(new_theta.values())
            if Z <= 0:
                print(f"  it {it}: Z <= 0, break")
                break
            new_theta = {i: v / Z for i, v in new_theta.items()}

            delta = sum(abs(new_theta[i] - theta[i]) for i in items)
            print(f"  it {it}: {[f'{new_theta[i]:.6f}' for i in items]}  delta={delta:.2e}")
            theta = new_theta
            if delta < 1e-9:
                break

        print(f"  w_i: {w_i}")
        print(f"  denom at last iter: {[f'{denom[i]:.2f}' for i in items]}")


# ============================================================================
# Section 4 — final theta_hat and rankings
# ============================================================================

def inspect_final_theta(learner, true_theta_men, true_theta_women):
    print("\n" + "=" * 78)
    print("SECTION 4 — Final theta_hat vs truth")
    print("=" * 78)

    for state in learner.agent_states.values():
        theta_vals = list(state.theta.values())
        rng = max(theta_vals) - min(theta_vals)

        print(f"\n[{state.agent}]")
        print(f"  θ̂ = {dict((str(k), round(v, 6)) for k, v in state.theta.items())}")
        print(f"  θ̂ range = {rng:.8f}")

        rank_hat = bt_ranking(state.theta)
        rank_true = truth_ranking(state, true_theta_men, true_theta_women)

        print(f"  ≻̂ = {[str(x) for x in rank_hat]}")
        print(f"  ≻  = {[str(x) for x in rank_true]}")
        print(f"  match = {rank_hat == rank_true}")

        if rank_hat != rank_true:
            if rng < 1e-6:
                print("  !! θ̂ is degenerate (range < 1e-6) — MLE not moving")
            else:
                print("  !! θ̂ has range but ranking is wrong — check sort key or θ̂ computation")


# ============================================================================
# Section 5 — arms orientation
# ============================================================================

def inspect_arms(learner):
    print("\n" + "=" * 78)
    print("SECTION 5 — Arm orientation vs canonical key")
    print("=" * 78)

    flipped = []
    for (agent, key, b1, b2) in learner._arms:
        if key[0] != b1:
            flipped.append((agent, key, b1, b2))

    print(f"Total arms: {len(learner._arms)}")
    print(f"Flipped orientation (key[0] != b1): {len(flipped)}")
    for entry in flipped[:10]:
        agent, key, b1, b2 = entry
        print(f"  agent={agent}, key=({key[0]},{key[1]}), "
              f"loop order=(b1={b1}, b2={b2})")


# ============================================================================
# Section 6 — manual test of bt_mle_mm on synthetic data
# ============================================================================

def test_bt_mle_mm_on_synthetic():
    print("\n" + "=" * 78)
    print("SECTION 6 — bt_mle_mm on synthetic data (known ground truth)")
    print("=" * 78)

    # Create 3 partners with known theta
    a, b, c = Woman("wa"), Woman("wb"), Woman("wc")
    items = [a, b, c]
    true_theta = {a: 0.5, b: 0.3, c: 0.2}

    # Simulate 1000 comparisons per pair with the true BT probabilities
    rng = np.random.default_rng(0)
    totals = {}
    wins = {}
    for x, y in [(a, b), (a, c), (b, c)]:
        key = _canonical(x, y)
        p_canonical_first_wins = (
            true_theta[key[0]] / (true_theta[key[0]] + true_theta[key[1]])
        )
        n = 1000
        w = int(rng.binomial(n, p_canonical_first_wins))
        totals[key] = n
        wins[key] = w

    print(f"  totals = {dict(totals)}")
    print(f"  wins   = {dict(wins)}")

    theta_hat = bt_mle_mm(items, wins, totals, max_iter=50, tol=1e-6)
    print(f"  true theta = {true_theta}")
    print(f"  θ̂         = {theta_hat}")

    # Normalise true theta for comparison
    Z = sum(true_theta.values())
    true_norm = {k: v / Z for k, v in true_theta.items()}
    print(f"  true (norm) = {true_norm}")

    # Ranking check
    order_hat = [str(x) for x in bt_ranking(theta_hat)]
    order_true = [str(x) for x in sorted(items, key=lambda i: -true_norm[i])]
    print(f"  ≻̂ = {order_hat}")
    print(f"  ≻  = {order_true}")
    print(f"  match = {order_hat == order_true}")


# ============================================================================
# Main
# ============================================================================

def main():
    N, K, seed, alpha = 3, 3, 0, 0.5

    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    men = [Man(f"m{i}") for i in range(1, N + 1)]
    women = [Woman(f"w{j}") for j in range(1, K + 1)]

    true_theta_men = random_theta(men, women, np_rng, alpha)
    true_theta_women = random_theta(women, men, np_rng, alpha)

    print("=" * 78)
    print("True thetas")
    print("=" * 78)
    for m in men:
        print(f"  {m}: {dict((str(k), round(v, 4)) for k, v in true_theta_men[m].items())}")
    for w in women:
        print(f"  {w}: {dict((str(k), round(v, 4)) for k, v in true_theta_women[w].items())}")

    h_star = gs_on_true_preferences(men, women, true_theta_men, true_theta_women)
    print(f"\nH_* pairs: {[(p.man.id, p.woman.id) for p in h_star.pairs]}")

    learner = P2ETG(
        men, women,
        true_theta_men=true_theta_men,
        true_theta_women=true_theta_women,
        rng=rng,
    )
    result = learner.run_until_stop(
        max_epochs=40,
        adaptive=True,
        check_every=500,
        max_samples=200_000,
        verbose=False,
    )

    print(f"\nStopped: {result['stopped']}")
    print(f"T_stop:  {result['T_stop']}")

    # Run all sections
    inspect_counts(learner)
    inspect_mle_inputs(learner)
    trace_mle(learner, max_iter=5)
    inspect_final_theta(learner, true_theta_men, true_theta_women)
    inspect_arms(learner)
    test_bt_mle_mm_on_synthetic()

    # Final summary
    print("\n" + "=" * 78)
    print("SUMMARY")
    print("=" * 78)
    print(f"  H_t == H_*: {result['matching'] == h_star}")
    for state in learner.agent_states.values():
        rank_hat = bt_ranking(state.theta)
        rank_true = truth_ranking(state, true_theta_men, true_theta_women)
        ok = rank_hat == rank_true
        print(f"  {state.agent}: ≻̂ == ≻ ? {ok}")


if __name__ == "__main__":
    main()

True thetas
  Man(m1): {'Woman(w1)': 0.3063, 'Woman(w2)': 0.0012, 'Woman(w3)': 0.6925}
  Man(m2): {'Woman(w1)': 0.2286, 'Woman(w2)': 0.1771, 'Woman(w3)': 0.5942}
  Man(m3): {'Woman(w1)': 0.3921, 'Woman(w2)': 0.2012, 'Woman(w3)': 0.4067}
  Woman(w1): {'Man(m1)': 0.1515, 'Man(m2)': 0.0014, 'Man(m3)': 0.8471}
  Woman(w2): {'Man(m1)': 0.2698, 'Man(m2)': 0.3628, 'Man(m3)': 0.3674}
  Woman(w3): {'Man(m1)': 0.0344, 'Man(m2)': 0.5211, 'Man(m3)': 0.4445}

H_* pairs: [('m2', 'w3'), ('m1', 'w2'), ('m3', 'w1')]

Stopped: False
T_stop:  200000

SECTION 1 — Raw counts and win keys per agent

[Man(m1)]
  totals: {(Woman(w1), Woman(w3)): 11111, (Woman(w3), Woman(w2)): 11111, (Woman(w1), Woman(w2)): 11111}
  wins:   {(Woman(w1), Woman(w3)): 3360, (Woman(w3), Woman(w2)): 26, (Woman(w1), Woman(w2)): 11070}
  totals - wins: set()
  wins - totals: set()

[Man(m2)]
  totals: {(Woman(w1), Woman(w2)): 11111, (Woman(w3), Woman(w2)): 11111, (Woman(w1), Woman(w3)): 11111}
  wins:   {(Woman(w1), Woman(w2)): 6131,